# Tests: `fasterai.quantize.fake_quantizer` (source `nbs/quantize/fake_quantizer.ipynb`)

In [ ]:
from fastcore.test import *
import contextlib, copy, io
import torch
import torch.nn as nn
from fastai.data.core import DataLoaders, Datasets
from torch.utils.data import TensorDataset
from fasterai.core.criteria import large_final
from fasterai.core.precision import FakeQuantSpec, fake_quant_spec
from fasterai.sparse.sparsifier import Sparsifier
from fasterai.quantize.fake_quantizer import *
from fasterai.quantize.fake_quantizer import _EPS, _fake_quantize, _qparams, _qrange

In [ ]:
torch.manual_seed(0)  # the fixtures below, and every draw after them, are reproducible

def _model():
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(8, 4, 3, padding=1), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(4, 2))

def _rel(a, b): return ((a - b).norm() / b.norm()).item()

def _dls_of(x):
    "A fastai `DataLoaders` over one batch of `x` — the only calibration input this class accepts"
    ds = TensorDataset(x, torch.zeros(len(x), dtype=torch.long))
    return DataLoaders.from_dsets(ds, ds, bs=len(x), device='cpu')

_X = torch.randn(4, 3, 8, 8)
_DLS = _dls_of(_X)

## The arithmetic

In [ ]:
# every width rounds onto a real grid: 16 bits is a 16-bit grid, not a pass-through
torch.manual_seed(0)
_w = torch.randn(8, 16)
_err = {b: (_fake_quantize(_w, b, True, 'per_tensor') - _w).abs().max().item() for b in (2, 4, 8, 16)}
assert _err[2] > _err[4] > _err[8] > _err[16] > 0, _err
test_ne(_fake_quantize(_w, 16, True, 'per_tensor').tolist(), _w.tolist())

In [ ]:
# channels spanning two orders of magnitude: per-tensor loses the quiet ones, per-channel keeps them.
# Row by row, per-channel is not ALWAYS closer (5 % of draws give a loud row a luckier global scale),
# so the claims here are the exact ones: the quiet rows, and the average over rows.
_spread = torch.randn(4, 64) * torch.tensor([1., 100., 1., 100.])[:, None]
_q_t = _fake_quantize(_spread, 4, True, 'per_tensor')
_q_c = _fake_quantize(_spread, 4, True, 'per_channel')
_rel_t = (_q_t - _spread).norm(dim=1) / _spread.norm(dim=1)
_rel_c = (_q_c - _spread).norm(dim=1) / _spread.norm(dim=1)

# one scale for the whole tensor is set by the loud rows, so the quiet ones round away entirely
test_eq(bool((_q_t[0] == 0).all() and (_q_t[2] == 0).all()), True)
test_close(_rel_t[0].item(), 1.0, eps=1e-6)      # rounded to zero is exactly 100 % of the row
test_eq(bool((_q_c[0] != 0).any() and (_q_c[2] != 0).any()), True)
assert _rel_c[0] < _rel_t[0] and _rel_c[2] < _rel_t[2], (_rel_c, _rel_t)
assert _rel_c.mean() < _rel_t.mean(), (_rel_c.mean(), _rel_t.mean())

In [ ]:
# a group covering the whole row is exactly the per-channel axis
_group = _fake_quantize(_spread, 4, True, 'per_group', group_size=_spread.shape[1])
test_eq(torch.equal(_group, _fake_quantize(_spread, 4, True, 'per_channel')), True)
# a direct caller gets the same refusal a model does, naming the row length
test_fail(lambda: _fake_quantize(_spread, 4, True, 'per_group', group_size=7), contains='the 64 weights')

In [ ]:
# symmetric puts every zero-point at 0; affine offsets a two-sided range
test_eq(_qparams(torch.randn(3, 40), 8, True, per_row=True)[1].tolist(), [0, 0, 0])
assert (_qparams(torch.randn(3, 40), 8, False, per_row=True)[1] > 0).all()
test_eq(_qrange(8, True), (-128, 127))
test_eq(_qrange(8, False), (0, 255))

In [ ]:
# a dead channel must not produce a zero scale, on either grid
for _symmetric in (True, False):
    _scale, _zero = _qparams(torch.zeros(2, 4), 8, _symmetric, per_row=True)
    assert (_scale >= _EPS).all(), (_symmetric, _scale)
    assert not _zero.isnan().any(), (_symmetric, _zero)
    test_eq(torch.equal(_fake_quantize(torch.zeros(2, 4), 8, _symmetric, 'per_channel'), torch.zeros(2, 4)), True)
_dead = torch.zeros(2, 4); _dead[0, 0] = 3.
assert (_qparams(_dead, 8, True, per_row=True)[0] > 0).all()

## What the grammar refuses

In [ ]:
test_fail(lambda: FakeQuantizer(_model(), 1), contains='[2, 16]')
test_fail(lambda: FakeQuantizer(_model(), 8.0), contains='[2, 16]', exc=TypeError)
test_fail(lambda: FakeQuantizer(_model(), True), contains='[2, 16]', exc=TypeError)
test_fail(lambda: FakeQuantizer(_model(), 8, 17), contains='[2, 16]')
test_fail(lambda: FakeQuantizer(_model(), 8, qscheme='per_axis'), contains='per_channel')
test_fail(lambda: FakeQuantizer(_model(), 8, qscheme=4), contains='(str)', exc=TypeError)
test_fail(lambda: FakeQuantizer(_model(), 8, observer='ema'), contains='dynamic')
# a per-layer width is validated too, and the message names the key it came from
test_fail(lambda: FakeQuantizer(_model(), 8, layer_bits={'0': 1}), contains="layer_bits['0']=1")

In [ ]:
test_fail(lambda: FakeQuantizer(_model(), 4, qscheme='per_group'), contains='group_size')
test_fail(lambda: FakeQuantizer(_model(), 4, qscheme='per_group', group_size=0), contains='at least one weight')
test_fail(lambda: FakeQuantizer(_model(), 4, qscheme='per_group', group_size=1.5),
          contains='positive int', exc=TypeError)
test_fail(lambda: FakeQuantizer(_model(), 4, group_size=64), contains="per_group")
# a row of 27 weights is not divisible by 8: refused at construction, naming the layer, before any write
test_fail(lambda: FakeQuantizer(_model(), 4, qscheme='per_group', group_size=8),
          contains="the 27 weights of a row of '0'")
# ... unless that layer was asked to stay in floating point
FakeQuantizer(_model(), 4, qscheme='per_group', group_size=8,
              layer_bits={'0': None, '2': None, '5': None})

In [ ]:
test_fail(lambda: FakeQuantizer(nn.Sequential(nn.ReLU()), 8), contains='layer_type')
test_fail(lambda: FakeQuantizer(_model(), 8, layer_type=nn.Embedding), contains='layer_type')

In [ ]:
# ConvTranspose weights are [in, out/groups, *kernel], so dim 0 is the input axis
_ct = nn.Sequential(nn.ConvTranspose2d(4, 8, 3))
test_fail(lambda: FakeQuantizer(_ct, 8, layer_type=nn.ConvTranspose2d), contains="per_tensor")
test_fail(lambda: FakeQuantizer(nn.Sequential(nn.ConvTranspose1d(4, 8, 3)), 8,
                                layer_type=nn.ConvTranspose1d), contains="*kernel")
_fq_ct = FakeQuantizer(_ct, 8, qscheme='per_tensor', layer_type=nn.ConvTranspose2d)
_fq_ct.quantize_model()
test_eq(fake_quant_spec(_ct).label, 'W8AF')

In [ ]:
# an Embedding and a ConvTranspose in a rounded model stay byte-identical under the default layer_type
_mixed = nn.ModuleDict({'emb': nn.Embedding(16, 8), 'up': nn.ConvTranspose2d(4, 4, 3),
                        'conv': nn.Conv2d(3, 8, 3)})
_emb0 = _mixed['emb'].weight.detach().clone()
_up0 = _mixed['up'].weight.detach().clone()
_fq_mixed = FakeQuantizer(_mixed, 8)
_fq_mixed.quantize_model()
test_eq(torch.equal(_mixed['emb'].weight, _emb0), True)
test_eq(torch.equal(_mixed['up'].weight, _up0), True)
test_ne(_mixed['conv'].weight.tolist(), _mixed['conv']._fp_weight.tolist())
_out = io.StringIO()
with contextlib.redirect_stdout(_out): _fq_mixed.print_precision()
assert 'emb' not in _out.getvalue() and 'W8AF' in _out.getvalue(), _out.getvalue()

In [ ]:
# observer='static' without calibrate() names the call that fixes it
test_fail(FakeQuantizer(_model(), 8, 8).quantize_model, contains='calibrate(data)')
_fq_dyn = FakeQuantizer(_model(), 8, 8, observer='dynamic')  # dynamic needs no calibration at all
_fq_dyn.quantize_model()
test_eq(fake_quant_spec(_fq_dyn.model).observer, 'dynamic')

In [ ]:
_fq_twice = FakeQuantizer(_model(), 8)
_fq_twice.quantize_model()
test_fail(_fq_twice.quantize_model, contains='remove()', exc=RuntimeError)
# remove() before quantize_model() is a no-op that still returns the model
_m = _model()
_fq_early = FakeQuantizer(_m, 8)
test_is(_fq_early.remove(), _m)
test_eq([k for k in _m.state_dict() if k.startswith(('0._', '2._'))], [])

## A failure mid-round must not cost the floating-point weights

In [ ]:
import fasterai.quantize.fake_quantizer as _fqm

_m = _model()
_fp0 = _m[0].weight.detach().clone()
_orig, _calls = _fqm._fake_quantize, []
def _boom(t, *args, **kwargs):
    _calls.append(1)
    if len(_calls) == 2: raise RuntimeError('kernel exploded')
    return _orig(t, *args, **kwargs)

_fqm._fake_quantize = _boom
try:
    _fq_half = FakeQuantizer(_m, 8)
    test_fail(_fq_half.quantize_model, contains='kernel exploded', exc=RuntimeError)
    # the first layer was rounded, and its snapshot is still the floating-point weight
    test_eq(torch.equal(_m[0]._fp_weight, _fp0), True)
    test_ne(_m[0].weight.tolist(), _fp0.tolist())
finally:
    _fqm._fake_quantize = _orig

# a retry must not re-snapshot the already-rounded weight, and remove() must give the fp32 one back
_fq_half.quantize_model()
test_eq(torch.equal(_m[0]._fp_weight, _fp0), True)
test_eq(len(_fq_half._act_hooks), 0)
_fq_half.remove()
test_eq(torch.equal(_m[0].weight, _fp0), True)

## Per-layer widths

In [ ]:
# an unknown name warns and is skipped, the way `Sparsifier` reports one
_out = io.StringIO()
with contextlib.redirect_stdout(_out): _fq_warn = FakeQuantizer(_model(), 8, layer_bits={'nope': 4})
assert "Warning: Layer 'nope'" in _out.getvalue(), _out.getvalue()
test_eq(len(_fq_warn._weight_map), 3)
# a module the quantizer does not round is named by its type, not by a repr
_out = io.StringIO()
with contextlib.redirect_stdout(_out): FakeQuantizer(_model(), 8, layer_bits={nn.Conv2d(1, 1, 1): 4})
assert "Warning: Layer 'Conv2d'" in _out.getvalue(), _out.getvalue()

In [ ]:
# a layer named None keeps its floating-point weight, byte for byte
_m = _model()
_untouched = _m[2].weight.detach().clone()
_fq_pl = FakeQuantizer(_m, 8, layer_bits={'2': None, '5': 4})
_fq_pl.quantize_model()
test_eq(torch.equal(_m[2].weight, _untouched), True)
test_eq('_fp_weight' in _m[2]._buffers, False)
test_ne(_m[0].weight.tolist(), _m[0]._fp_weight.tolist())
test_eq(fake_quant_spec(_m).layer_bits, {'2': None, '5': 4})

In [ ]:
# a module key works like a name key, the way `Sparsifier` and `Pruner` accept one
_m = _model()
_fp = _m[2].weight.detach().clone()
FakeQuantizer(_m, 8, layer_bits={_m[2]: None}).quantize_model()
test_eq(torch.equal(_m[2].weight, _fp), True)
test_ne(_m[0].weight.tolist(), _m[0]._fp_weight.tolist())

## Calibration data shapes

In [ ]:
torch.manual_seed(0)
# a `DataLoaders` built the real way: its loaders delegate an unknown `.train` to the `Datasets`
_dsets = Datasets(list(range(16)), tfms=[[lambda i: torch.zeros(3, 8, 8) + i / 16], [lambda i: i % 2]],
                  splits=[list(range(8)), list(range(8, 16))])
_real = _dsets.dataloaders(bs=4, device='cpu', shuffle=False)

for _name, _data in {'fastai DataLoaders': _real, 'fastai DataLoader': _real.train,
                     'DataLoaders.from_dsets': _DLS}.items():
    _m = _model()
    _fq = FakeQuantizer(_m, 8, 8)
    _fq.calibrate(_data, n_batches=2)
    _fq.quantize_model()
    assert float(_m[0]._act_scale) > 0, _name
    test_eq(_m[0]._act_zero_point.dtype, torch.int32)

# a `DataLoaders` and the loader it holds must calibrate ONE model on the same batches: reading `.train`
# off the loader would reach its `Datasets` and measure unbatched items instead
_m = _model()
_fq = FakeQuantizer(_m, 8, 8)
_fq.calibrate(_real, n_batches=2);       _from_container = float(_m[0]._act_scale)
_fq.calibrate(_real.train, n_batches=2); _from_loader = float(_m[0]._act_scale)
test_close(_from_container, _from_loader, eps=1e-9)

In [ ]:
# training and validation go through fastai, so calibration does too: everything else is refused by name
from torch.utils.data import DataLoader as _TorchDL

for _wrong in (_TorchDL(TensorDataset(_X, torch.zeros(4, dtype=torch.long)), batch_size=2),
               [(torch.randn(4, 3, 8, 8),)], _X):
    test_fail(lambda: FakeQuantizer(_model(), 8, 8).calibrate(_wrong), exc=TypeError,
              contains='must be a fastai `DataLoaders` or one of its loaders')
    test_fail(lambda: FakeQuantizer(_model(), 8, 8).calibrate(_wrong), exc=TypeError,
              contains='pass `dls` or `dls.train`')
test_fail(lambda: FakeQuantizer(_model(), 8, 8).calibrate(_X), contains='Tensor', exc=TypeError)

class _FakeLoader:
    "Something in the fastai loader shape, so the batches themselves are what gets refused"
    def __init__(self, batches): self.batches = batches
    def __iter__(self): return iter(self.batches)
    def one_batch(self): return self.batches[0]

test_fail(lambda: FakeQuantizer(_model(), 8, 8).calibrate(_FakeLoader([('not a tensor',)])),
          contains='must yield tensors', exc=TypeError)
test_fail(lambda: FakeQuantizer(_model(), 8, 8).calibrate(_FakeLoader([])), contains='at least one batch')
test_fail(lambda: FakeQuantizer(_model(), 8).calibrate(_DLS), contains='act_bits')

# a failed calibration leaves no observation state behind, on a REUSED model
_m = _model()
_fq_fail = FakeQuantizer(_m, 8, 8)
test_fail(lambda: _fq_fail.calibrate(_FakeLoader([('not a tensor',)])), contains='yield tensors', exc=TypeError)
test_fail(lambda: _fq_fail.calibrate(_FakeLoader([])), contains='at least one batch')
test_eq([k for k in _m.state_dict() if '_act' in k], [])
test_eq([n for n, _ in _m.named_buffers() if '_act' in n], [])
_fq_fail.calibrate(_DLS)          # and the model still calibrates afterwards
_fq_fail.quantize_model()
assert float(_m[0]._act_scale) > 0

In [ ]:
# n_batches really limits what is observed. One model, copied per arm, and no bias: the output is then
# exactly linear in the input, so observing the ramp 1,2 against 1,2,3,4 must give exactly twice the
# range — a property no draw can move, where "merely wider" flaked on 0.3 % of seeds.
_widening = _FakeLoader([(torch.full((2, 3, 8, 8), float(i)),) for i in range(1, 5)])
_base_wide = nn.Sequential(nn.Conv2d(3, 8, 3, bias=False))

_scales = []
for _n in (2, 4):
    _m = copy.deepcopy(_base_wide)
    _fq = FakeQuantizer(_m, 8, 8)
    _fq.calibrate(_widening, n_batches=_n)
    _fq.quantize_model()
    _scales.append(float(_m[0]._act_scale))
test_close(_scales[1] / _scales[0], 2.0, eps=1e-5)

In [ ]:
# recalibrating an already-rounded model is supported: the rounding hooks stand down while observing
_m = _model()
_fq_re = FakeQuantizer(_m, 8, 8)
_fq_re.calibrate(_DLS)
_fq_re.quantize_model()
_first = float(_m[0]._act_scale)
_fq_re.calibrate(_dls_of(_X * 10))
_second = float(_m[0]._act_scale)

def _observed_scale(x):
    "The range this layer's output really spans, over the largest integer of a symmetric 8-bit grid"
    for _, _r in _fq_re._act_hooks.values(): _r.enabled = False
    with torch.no_grad(): _out = _m[0](x)
    for _, _r in _fq_re._act_hooks.values(): _r.enabled = True
    return max(_out.max().item(), -_out.min().item()) / 127

# the second calibration froze the range of the NEW data. Not a ratio: this layer has a bias, so its
# output is W.x + b and a tenfold input does not give a tenfold range.
test_close(_second, _observed_scale(_X * 10), eps=1e-6)
assert _observed_scale(_X * 10) != _observed_scale(_X), 'degenerate fixture: both inputs span one range'
test_ne(_second, _first)

## Lifecycle: quantize, save, copy, remove

In [ ]:
_m = _model()
_ref = _m(_X).detach().clone()
_before = {n: p.detach().clone() for n, p in _m.named_parameters()}
_keys = list(_m.state_dict())
_fq = FakeQuantizer(_m, 8, 8)
_fq.calibrate(_DLS)
_fq.quantize_model()
_rounded = _m(_X).detach().clone()
test_ne(_rounded.tolist(), _ref.tolist())

# the rounded state dict carries exactly the keys it had: every buffer here is non-persistent
test_eq(list(_m.state_dict()), _keys)
assert '_fp_weight' in _m[0]._buffers and float(_m[0]._act_scale) > 0

# the point of writing into m.weight.data instead of parametrizing it
_copy = copy.deepcopy(_m)
test_eq(torch.equal(_copy(_X), _rounded), True)
_buf = io.BytesIO(); torch.save(_m, _buf)
_copy[0]._act_scale.mul_(2)      # a copy must own its calibration, not share it
test_ne(float(_copy[0]._act_scale), float(_m[0]._act_scale))

_fq.remove()
test_eq(all(torch.equal(p, _before[n]) for n, p in _m.named_parameters()), True)
test_eq(torch.equal(_m(_X), _ref), True)
test_eq(fake_quant_spec(_m), None)
test_eq([n for n, _ in _m.named_buffers()], [])

In [ ]:
_m = nn.Sequential(nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 4))
_fq = FakeQuantizer(_m, 4, 8, qscheme='per_group', group_size=8, observer='dynamic', symmetric=False)
_spec = _fq.quantize_model() and fake_quant_spec(_m)
test_eq(_spec, _fq.spec)
test_eq(_spec.label, 'W4A8')
test_eq((_spec.weight_bits, _spec.act_bits, _spec.qscheme, _spec.group_size), (4, 8, 'per_group', 8))
test_eq((_spec.symmetric, _spec.observer), (False, 'dynamic'))
test_eq(_spec.as_dict()['qscheme'], 'per_group')

# rounding a trained model is not training through the rounding: `trained` is False on every path here,
# and only `FakeQuantizeCallback` sets it
test_eq(_spec.trained, False)
test_eq(_spec.as_dict()['trained'], False)
for _make in (lambda m: FakeQuantizer(m, 8),
              lambda m: FakeQuantizer(m, 4, qscheme='per_tensor'),
              lambda m: FakeQuantizer(m, 8, 8, observer='dynamic')):
    _p = _model()
    _make(_p).quantize_model()
    test_eq(fake_quant_spec(_p).trained, False)
_p = _model()
_fq_calib = FakeQuantizer(_p, 8, 8)
_fq_calib.calibrate(_DLS)
_fq_calib.quantize_model()
test_eq(fake_quant_spec(_p).trained, False)

## What the report may claim

In [ ]:
# a rounded model is not smaller and not faster: nothing here may suggest otherwise
_FORBIDDEN = ('faster', 'speedup', 'smaller', 'compression', 'size', 'shrink', 'reduce')
_docs = {'FakeQuantizer': FakeQuantizer.__doc__, 'FakeQuantSpec': FakeQuantSpec.__doc__,
         **{n: getattr(FakeQuantizer, n).__doc__ for n in
            ('calibrate', 'quantize_model', 'remove', 'print_precision')},
         'spec': FakeQuantizer.spec.__doc__}
assert 'simulated' in _docs['FakeQuantizer'].lower(), _docs['FakeQuantizer']
for _name, _doc in _docs.items():
    for _word in _FORBIDDEN:
        test_eq((_name, _word in _doc.lower()), (_name, False))

_m = _model()
_fq = FakeQuantizer(_m, 8, layer_bits={'5': 4})
_fq.quantize_model()
_out = io.StringIO()
with contextlib.redirect_stdout(_out): _fq.print_precision()
_report = _out.getvalue().lower()
for _word in _FORBIDDEN + ('ratio',):
    test_eq((_word, _word in _report), (_word, False))
assert '4 bits' in _report and '8 bits' in _report, _report

## Composition with `Sparsifier`

In [ ]:
# zero quantizes to the zero-point and dequantizes to exactly 0, on both grids
for _symmetric in (True, False):
    _m = _model()
    _sp = Sparsifier(_m, 'weight', 'local', large_final, layer_type=nn.Conv2d)
    _sp.sparsify_model(0.5)
    _zeros = [(m.weight == 0).sum().item() for m in _m.modules() if isinstance(m, nn.Conv2d)]
    FakeQuantizer(_m, 8, symmetric=_symmetric).quantize_model()
    _after = [(m.weight == 0).sum().item() for m in _m.modules() if isinstance(m, nn.Conv2d)]
    test_eq(_after, _zeros)
    assert min(_zeros) > 0, _zeros

## Integration (`#| slow`)

In [ ]:
#| slow
from torchvision.models import resnet18

torch.manual_seed(0)
_xr = torch.randn(4, 3, 64, 64)
_base = resnet18(weights=None).eval()
with torch.no_grad(): _refr = _base(_xr)

def _round_resnet(weight_bits, act_bits, **kwargs):
    m = copy.deepcopy(_base)
    fq = FakeQuantizer(m, weight_bits, act_bits, **kwargs)
    if act_bits is not None: fq.calibrate(_dls_of(_xr))
    fq.quantize_model()
    with torch.no_grad(): return fq, m, _rel(m(_xr), _refr)

_fq8, _m8, _err8 = _round_resnet(8, 8)
_fq4, _m4, _err4 = _round_resnet(4, 8)
_, _, _err_a4 = _round_resnet(8, 4)
print(f'W8A8 {_err8:.4f}  W4A8 {_err4:.4f}  W8A4 {_err_a4:.4f}')
assert _err8 < 0.10, _err8          # W8A8 stays close to fp32 on this batch
assert _err4 > _err8, (_err4, _err8) # a narrower weight costs more, an ordering not a number
assert _err_a4 > _err8, (_err_a4, _err8)

In [ ]:
#| slow
# remove() restores the fp weights bit for bit, and a rounded ResNet still saves and copies
_fp = {n: p.detach().clone() for n, p in _base.named_parameters()}  # `_m8` is a copy of `_base`
_keys = list(_base.state_dict())
test_eq(list(_m8.state_dict()), _keys)   # rounding added no state dict key
_copy = copy.deepcopy(_m8)
_buf = io.BytesIO(); torch.save(_m8, _buf)
with torch.no_grad(): test_eq(torch.equal(_copy(_xr), _m8(_xr)), True)
_fq8.remove()
test_eq(all(torch.equal(p, _fp[n]) for n, p in _m8.named_parameters()), True)
with torch.no_grad(): test_eq(torch.equal(_m8(_xr), _refr), True)
test_eq([n for n, _ in _m8.named_buffers() if '_fp_weight' in n or '_act_' in n], [])

In [ ]:
#| slow
# a per-layer dict leaving one layer in floating point leaves exactly that layer alone
_mp = copy.deepcopy(_base)
_fp_layer = _mp.layer4[1].conv2.weight.detach().clone()
_narrow = _mp.layer1[0].conv1.weight.detach().clone()
FakeQuantizer(_mp, 8, layer_bits={'layer4.1.conv2': None, 'layer1.0.conv1': 4}).quantize_model()
test_eq(torch.equal(_mp.layer4[1].conv2.weight, _fp_layer), True)
test_ne(_mp.layer1[0].conv1.weight.tolist(), _narrow.tolist())
# per_channel: each output channel carries its own 4-bit grid
_rows = _mp.layer1[0].conv1.weight.flatten(1)
test_eq(max(len(r.unique()) for r in _rows) <= 2 ** 4, True)

In [ ]:
#| slow
# the model runs where it was trained: CPU above, CUDA here when the box has one
if torch.cuda.is_available():
    _mc = copy.deepcopy(_base).cuda()
    _xc = _xr.cuda()
    with torch.no_grad(): _refc = _mc(_xc)
    _fqc = FakeQuantizer(_mc, 8, 8)
    _fqc.calibrate(_dls_of(_xr))   # CPU batches, CUDA model: calibrate() moves each batch itself
                                   # (a `DataLoaders` over CUDA tensors cannot fork its workers)
    _fqc.quantize_model()
    with torch.no_grad(): _errc = _rel(_mc(_xc), _refc)
    print(f'cuda W8A8 {_errc:.4f}')
    assert _errc < 0.10, _errc
    test_eq(_mc.conv1._act_scale.device.type, 'cuda')
    _mg = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10)).cuda()
    FakeQuantizer(_mg, 4, qscheme='per_group', group_size=32).quantize_model()
    test_eq(fake_quant_spec(_mg).label, 'W4AF')
else:
    print('no CUDA on this box, CPU arms only')

In [ ]:
#| slow
# the user-side path: calibrate straight off a fastai `DataLoaders`, then validate
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from torch.utils.data import TensorDataset

torch.manual_seed(0)
_ds = TensorDataset(torch.randn(32, 3, 32, 32), torch.randint(0, 10, (32,)))
_dls = DataLoaders.from_dsets(_ds, _ds, bs=8, device='cpu')
_net = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                     nn.Flatten(), nn.Linear(8, 10))
_learn = Learner(_dls, _net, loss_func=nn.CrossEntropyLoss())
_loss_fp = _learn.validate()[0]
_fql = FakeQuantizer(_net, 8, 8)
_fql.calibrate(_dls)
_fql.quantize_model()
_loss_q = _learn.validate()[0]
print(f'fp32 {_loss_fp:.4f}  W8A8 {_loss_q:.4f}')
test_close(_loss_q, _loss_fp, eps=0.5)
_fql.remove()
test_close(_learn.validate()[0], _loss_fp, eps=1e-6)